<a href="https://colab.research.google.com/github/nataliehany/FlyRank-ML-Internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Vinay21rout/flyrank-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("HF_TOKEN loaded successfully:", HF_TOKEN is not None)

HF_TOKEN loaded successfully: True


In [ ]:
from google.colab import userdata
from huggingface_hub import whoami

HF_TOKEN = userdata.get("HF_TOKEN")

user = whoami(token=HF_TOKEN)
print("Logged into Hugging Face as:", user["name"])

Logged into Hugging Face as: nataliehany


In [ ]:
from huggingface_hub import list_repo_files
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

files = list_repo_files(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    token=HF_TOKEN
)

print("Dataset access successful!")
print("Number of files:", len(files))
print("\nFirst files:")
for f in files[:20]:
    print(f)

Dataset access successful!
Number of files: 24

First files:
.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/mon

In [ ]:
from huggingface_hub import hf_hub_download
from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")

march_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

con = duckdb.connect()

print("=== COLUMNS AND DATA TYPES ===")
display(
    con.execute(
        "DESCRIBE SELECT * FROM read_parquet(?)",
        [march_file]
    ).df()
)

print("\n=== FIRST 5 ROWS ===")
display(
    con.execute(
        "SELECT * FROM read_parquet(?) LIMIT 5",
        [march_file]
    ).df()
)

=== COLUMNS AND DATA TYPES ===


,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None



=== FIRST 5 ROWS ===


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Data contract

**One row:** One content item for one pseudonymized client on one reporting date.

**Table used:** `fact_content_daily_performance`

**Time window:** March 1–31, 2026 (`month = '2026-03'`). I use a mid-panel month so the final month remains separate from development.

**Prediction / ranking goal:** Rank content items by refresh opportunity — identifying content that receives search visibility but has relatively weak click performance and may benefit from content improvement.

**Deliberately excluded:** Client and content identifiers are used only to define the observation grain, not as predictive features. They identify entities rather than describing content performance.

In [ ]:
march_info = con.execute("""
    SELECT
        COUNT(*) AS row_count
    FROM read_parquet(?)
    WHERE month = '2026-03'
""", [march_file]).df()

column_count = len(
    con.execute(
        "DESCRIBE SELECT * FROM read_parquet(?)",
        [march_file]
    ).df()
)

print("March 2026 data accessible successfully.")
print("Rows:", march_info.loc[0, "row_count"])
print("Columns:", column_count)

March 2026 data accessible successfully.
Rows: 9841378
Columns: 31


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Field roles

**Features — exactly five**

1. `gsc_impressions` — measures how often the content appeared in Google Search results.
2. `gsc_clicks` — measures observed search clicks to the content.
3. `gsc_avg_position` — measures the content's average Google Search ranking position.
4. `ga4_pageviews` — measures observed page-view activity when GA4 data is available.
5. `ga4_engaged_sessions` — measures engaged visits when GA4 data is available.

These features describe observed content performance and are candidates for ranking content refresh opportunities.

**Label / proxy**

The modeling target will be a refresh-opportunity proxy constructed separately from the feature frame. The goal is to rank content items that show search visibility but relatively weak subsequent performance. I will not use a label-derived value as an honest input feature.

**Context fields**

- `report_date` — defines when the observation was measured.
- `month` — restricts development to March 2026.
- `client_hash_id` — identifies the pseudonymized client and helps define the grain.
- `content_hash_id` — identifies the pseudonymized content item and helps define the grain.
- `gsc_data_available` — indicates whether GSC observations are available.
- `ga4_data_available` — indicates whether GA4 observations are available.

**Excluded**

- `client_hash_id` and `content_hash_id` are excluded from predictive features because they are identifiers, not performance measurements.
- Raw channel/platform breakdown fields such as `ai_chatgpt`, `ai_perplexity`, `ai_gemini`, and other traffic-source columns are excluded to keep this first feature frame intentionally small.
- Any feature calculated directly from the target/label is excluded from the honest model because it would create target leakage.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
feature_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "gsc_ctr",
    "position_opportunity"
]

print("Number of planned features:", len(feature_cols))
context_cols = [
    "report_date",
    "month",
    "client_hash_id",
    "content_hash_id",
    "gsc_data_available",
    "ga4_data_available"
]

print("Feature columns:")
for col in feature_cols:
    print("-", col)

print("\nNumber of features:", len(feature_cols))

print("\nContext columns:")
for col in context_cols:
    print("-", col)

Number of planned features: 5
Feature columns:
- gsc_impressions
- gsc_clicks
- gsc_avg_position
- gsc_ctr
- position_opportunity

Number of features: 5

Context columns:
- report_date
- month
- client_hash_id
- content_hash_id
- gsc_data_available
- ga4_data_available


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Verification plan

I use exactly three verification queries on March 2026:

1. **Grain check** — verify whether each row is unique at `report_date × client_hash_id × content_hash_id`.
2. **Slice size and date span** — measure the number of rows and confirm the observed March 2026 date range.
3. **Availability check** — use `IS TRUE` to measure how many observations have the source data needed for the candidate features.

The results below are measured from the warehouse rather than assumed from the schema.

### Five-feature frame and decision moment

**Decision moment:** At the end of a content item's current reporting day, use information observed on that day to estimate whether the same content item will receive at least one Google Search click on its next observed day.

**Label:** `next_day_click` — 1 if the content item has at least one GSC click on its next observed reporting day, otherwise 0.

The five features are:

1. `gsc_impressions` — **available when?** Known by the end of the current reporting day because it is measured from current-day GSC data.
2. `gsc_clicks` — **available when?** Known by the end of the current reporting day because it is a current-day observed GSC metric.
3. `gsc_avg_position` — **available when?** Known by the end of the current reporting day from current-day search-position observations.
4. `gsc_ctr` — **available when?** Computed from current-day clicks and impressions, so it is knowable at the same decision moment.
5. `has_click_today` — **available when?** Derived only from the current day's `gsc_clicks`, so it is known before the next-day outcome occurs.

The label is kept separate from these five features because it comes from the next observed day.

In [ ]:
feature_frame_query = """
WITH gsc_rows AS (
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,

        CASE
            WHEN gsc_impressions > 0
            THEN CAST(gsc_clicks AS DOUBLE) / gsc_impressions
            ELSE 0.0
        END AS gsc_ctr,

        CASE
            WHEN gsc_clicks > 0 THEN 1
            ELSE 0
        END AS has_click_today,

        LEAD(gsc_clicks) OVER (
            PARTITION BY client_hash_id, content_hash_id
            ORDER BY report_date
        ) AS next_observed_clicks,

        LEAD(report_date) OVER (
            PARTITION BY client_hash_id, content_hash_id
            ORDER BY report_date
        ) AS next_observed_date

    FROM read_parquet(?)
    WHERE month = '2026-03'
      AND gsc_data_available IS TRUE
)

SELECT
    report_date,
    client_hash_id,
    content_hash_id,

    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    gsc_ctr,
    has_click_today,

    CASE
        WHEN next_observed_clicks > 0 THEN 1
        ELSE 0
    END AS next_day_click

FROM gsc_rows

WHERE next_observed_clicks IS NOT NULL
  AND next_observed_date = report_date + INTERVAL 1 DAY
"""

feature_frame = con.execute(
    feature_frame_query,
    [march_file]
).df()

print("Feature-frame rows:", len(feature_frame))

print("\nLabel distribution:")
display(
    feature_frame["next_day_click"]
    .value_counts()
    .rename_axis("next_day_click")
    .reset_index(name="rows")
)

print("\nPreview:")
display(feature_frame.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature-frame rows: 3194793

Label distribution:


,next_day_click,rows
0,0,2794544
1,1,400249



Preview:


,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,gsc_ctr,has_click_today,next_day_click
0,2026-03-25,client_0797ff3a1fc9a6a5,content_c88d7630a340a086,7,0,4.857143,0.0,0,0
1,2026-03-26,client_0797ff3a1fc9a6a5,content_c88d7630a340a086,35,0,8.171429,0.0,0,0
2,2026-03-27,client_0797ff3a1fc9a6a5,content_c88d7630a340a086,37,0,7.675676,0.0,0,0
3,2026-03-28,client_0797ff3a1fc9a6a5,content_c88d7630a340a086,43,0,7.465116,0.0,0,0
4,2026-03-29,client_0797ff3a1fc9a6a5,content_c88d7630a340a086,46,0,7.347826,0.0,0,0


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# Five honest features
honest_features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "gsc_ctr",
    "has_click_today"
]

target = "next_day_click"

# Keep only required columns and remove missing feature values
model_data = feature_frame[
    honest_features + [target]
].dropna().copy()

# Reproducible sample for a quick leakage demonstration
sample_n = min(300_000, len(model_data))

model_sample = model_data.sample(
    n=sample_n,
    random_state=42
)

X = model_sample[honest_features]
y = model_sample[target]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# -------------------------
# 1. Honest model
# -------------------------

honest_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=8,
    random_state=42,
    n_jobs=-1
)

honest_model.fit(X_train, y_train)

honest_prob = honest_model.predict_proba(X_test)[:, 1]

honest_auc = roc_auc_score(
    y_test,
    honest_prob
)

print("HONEST MODEL")
print("Features:", honest_features)
print("ROC-AUC:", round(honest_auc, 4))


# -------------------------
# 2. Deliberate leakage
# -------------------------

model_sample["leaked_next_day_click"] = model_sample[target]

leaked_features = honest_features + [
    "leaked_next_day_click"
]

X_leaked = model_sample[leaked_features]
y_leaked = model_sample[target]

X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(
    X_leaked,
    y_leaked,
    test_size=0.20,
    random_state=42,
    stratify=y_leaked
)

leaked_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=8,
    random_state=42,
    n_jobs=-1
)

leaked_model.fit(
    X_train_l,
    y_train_l
)

leaked_prob = leaked_model.predict_proba(
    X_test_l
)[:, 1]

leaked_auc = roc_auc_score(
    y_test_l,
    leaked_prob
)

print("\nLEAKED MODEL")
print("Added feature: leaked_next_day_click")
print("ROC-AUC:", round(leaked_auc, 4))


# -------------------------
# 3. Remove leakage
# -------------------------

model_sample = model_sample.drop(
    columns=["leaked_next_day_click"]
)

print("\nLEAKAGE REMOVED")
print(
    "leaked_next_day_click present:",
    "leaked_next_day_click" in model_sample.columns
)

print(
    "Honest ROC-AUC retained:",
    round(honest_auc, 4)
)

HONEST MODEL
Features: ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'gsc_ctr', 'has_click_today']
ROC-AUC: 0.8748

LEAKED MODEL
Added feature: leaked_next_day_click
ROC-AUC: 1.0

LEAKAGE REMOVED
leaked_next_day_click present: False
Honest ROC-AUC retained: 0.8748


### Leakage result

Using only the five features available at the decision moment, the quick model achieved an ROC-AUC of **0.8745**.

I then deliberately added `leaked_next_day_click`, which is copied directly from the future label. The ROC-AUC increased to **1.0000**. This apparently perfect performance is not a genuine modeling improvement: the feature contains the answer the model is supposed to predict.

After demonstrating the leakage, I removed `leaked_next_day_click`. The retained honest ROC-AUC is **0.8745**.

This experiment shows why feature availability must be evaluated relative to the prediction decision moment. A feature derived from information that becomes known only after the prediction point must not be used for training an honest model.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Named limitation: uneven source-data availability")
print("GSC available rows:", 3611061)
print("GA4 available rows:", 413966)
print("Both available rows:", 364347)
print("Development window: March 2026")
print("Honest ROC-AUC:", round(honest_auc, 4))

Named limitation: uneven source-data availability
GSC available rows: 3611061
GA4 available rows: 413966
Both available rows: 364347
Development window: March 2026
Honest ROC-AUC: 0.8748


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.